# CrashDiag SFT — independent Kaggle notebook

This notebook starts from a fresh Kaggle GPU session. It clones CrashDiag, reads only the `HF_TOKEN` Kaggle Secret, mechanically generates and uploads the four datasets, trains the LoRA SFT adapter, and uploads every retained checkpoint plus the final adapter/state/metrics to the private `devaanshpa/CrashDiag` Storage Bucket.

Enable **GPU** and **Internet** in Kaggle before running all cells. Record the printed `RUN_ID` and `SOURCE_COMMIT`; the independent GRPO notebook uses them to retrieve this notebook's artifacts with the exact same code revision. No Vultr sandbox credential is needed for SFT.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os
import secrets
import subprocess
import sys

REPO_URL = "https://github.com/Indium-AI-Labs/CrashDiag.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/CrashDiag")
BUCKET_ID = "devaanshpa/CrashDiag"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RUN_ID = os.environ.get("CRASHDIAG_RUN_ID") or (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "-sft-"
    + secrets.token_hex(3)
)
TRAIN_SAMPLES_PER_FAULT = 128
EVAL_SAMPLES_PER_FAULT = 16
SEED = 42
EPOCHS = 3.0
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
SAVE_STEPS = 25
PRECISION = "auto"

print(f"RUN_ID={RUN_ID}")
print(f"bucket={BUCKET_ID}")
print(f"base_model={BASE_MODEL}")

## Install the exact repository revision

This cell is safe in a fresh session and fast-forwards an existing clean clone. It never writes credentials into the repository.

In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{REPO_BRANCH}"], check=True)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"Refusing to overwrite non-repository directory: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train]"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(f"SOURCE_COMMIT={SOURCE_COMMIT}")

## Load the Kaggle Secret and verify the private bucket

Create a Kaggle Secret named `HF_TOKEN` with fine-grained write access to `devaanshpa/CrashDiag`. The value is placed only in process memory and is never displayed or written to notebook output.

In [ ]:
def required_secret(name: str) -> str:
    existing = os.environ.get(name)
    if existing:
        return existing
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Attach the Kaggle Secret {name!r} before continuing") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty")
    return value

os.environ["HF_TOKEN"] = required_secret("HF_TOKEN")
os.environ["CRASHDIAG_HF_BUCKET_ID"] = BUCKET_ID
os.environ["CRASHDIAG_ARTIFACT_PREFIX"] = "runs"
os.environ["CRASHDIAG_ARTIFACT_LOCAL_ROOT"] = str(REPO_DIR / "artifacts")
os.environ["CRASHDIAG_ARTIFACT_UPLOAD_POLICY"] = "required"
os.environ["CRASHDIAG_RUN_ID"] = RUN_ID

from training.artifacts import ArtifactConfig, ArtifactUploader

artifact_config = ArtifactConfig(
    bucket_id=BUCKET_ID,
    run_id=RUN_ID,
    prefix="runs",
    policy="required",
    local_root=REPO_DIR / "artifacts",
    token=os.environ["HF_TOKEN"],
)
uploader = ArtifactUploader(artifact_config)
uploader.start_run({"entrypoint": "notebooks/sft.ipynb", "base_model": BASE_MODEL, "source_commit": SOURCE_COMMIT})
print(f"private artifact run: {uploader.remote_uri()}")

## Verify the Kaggle GPU

Automatic precision selects BF16 only when the device supports it and otherwise uses FP16.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is visible. Enable a GPU in Kaggle Notebook Settings.")
print(f"torch={torch.__version__}")
print(f"gpu={torch.cuda.get_device_name(0)}")
print(f"bf16_supported={torch.cuda.is_bf16_supported()}")

## Generate and mechanically validate the datasets

Each SFT target is executed against a fresh deterministic `MockSandbox` and kept only when both the fault-specific resolution check and application health pass. GRPO rows contain the same prompt and replay seed but no answer. This cell uploads all four JSONL files, hashes, a manifest, and the dataset success marker.

In [ ]:
from training.generate_dataset import main as generate_dataset_main

dataset_exit = generate_dataset_main([
    "--train-samples-per-fault", str(TRAIN_SAMPLES_PER_FAULT),
    "--eval-samples-per-fault", str(EVAL_SAMPLES_PER_FAULT),
    "--seed", str(SEED),
])
if dataset_exit != 0:
    raise RuntimeError(f"dataset generation exited with {dataset_exit}")
for path in ("data/sft_train.jsonl", "data/sft_eval.jsonl", "data/grpo_train.jsonl", "data/grpo_eval.jsonl"):
    if not Path(path).is_file():
        raise RuntimeError(f"missing generated dataset: {path}")

## Run LoRA SFT

The notebook uses the unit-tested training backend directly in this single-GPU process. Every save is synchronously copied to the private bucket before training proceeds.

In [ ]:
from training.sft import main as sft_main

sft_exit = sft_main([
    "--model", BASE_MODEL,
    "--dataset", "data/sft_train.jsonl",
    "--eval-dataset", "data/sft_eval.jsonl",
    "--output-dir", "outputs/sft",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--precision", PRECISION,
    "--save-strategy", "steps",
    "--save-steps", str(SAVE_STEPS),
    "--save-total-limit", "2",
    "--report-to", "none",
])
if sft_exit != 0:
    raise RuntimeError(f"SFT exited with {sft_exit}")

## Verify the handoff to GRPO

Do not mark the whole run complete yet; GRPO and mechanical evaluation are separate stages. Copy the printed `RUN_ID` and `SOURCE_COMMIT` into the GRPO notebook.

In [ ]:
required_outputs = [
    REPO_DIR / "outputs/sft/adapter_config.json",
    REPO_DIR / "artifacts" / RUN_ID / "datasets/_SUCCESS.json",
    REPO_DIR / "artifacts" / RUN_ID / "sft/manifest.json",
    REPO_DIR / "artifacts" / RUN_ID / "sft/_SUCCESS.json",
]
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise RuntimeError(f"SFT handoff is incomplete: {missing}")
Path("/kaggle/working/crashdiag_handoff.txt").write_text(
    f"RUN_ID={RUN_ID}\nSOURCE_COMMIT={SOURCE_COMMIT}\n", encoding="utf-8"
)
print("SFT stage complete and uploaded.")
print(f"RUN_ID={RUN_ID}")
print(f"SOURCE_COMMIT={SOURCE_COMMIT}")
print(f"GRPO input: {uploader.remote_uri('sft')}")